In [1]:
import pandas as pd
import numpy as np
import os
import logging

In [2]:
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [3]:
Bureau = pd.read_csv('data/External/bureau.csv')

In [4]:
BureauBal = pd.read_csv('data/External/bureau_balance.csv')

In [11]:
BureauBal.dtypes

SK_ID_BUREAU       int64
MONTHS_BALANCE     int64
STATUS            object
dtype: object

In [6]:
Bureau = Bureau.astype({'SK_ID_CURR': 'string', 'SK_ID_BUREAU': 'string'})

Here is the correct way to interpret the variables for a credit card:

- AMT_CREDIT_SUM: This is the total credit limit on the card.

- AMT_CREDIT_SUM_DEBT: This is the current balance, or how much you owe.

- AMT_CREDIT_SUM_LIMIT: This is the available credit, or how much you can still spend.

- For closed account (~ 1m) `AMT_CREDIT_SUM_LIMIT` and `AMT_CREDIT_SUM_DEBT` = 0 (mostly) 
- For active accounts (~ 600k) 
    - Consumer Credits (~ 300k) have zero  `AMT_CREDIT_SUM_LIMIT`
    - Credit Cards (~ 300k) 
        - 40k have `AMT_CREDIT_SUM` > 0 and `AMT_CREDIT_SUM_DEBT`= 0 and `AMT_CREDIT_SUM_LIMIT`=0.
        - 87k have > `AMT_CREDIT_SUM` = `AMT_CREDIT_SUM_DEBT` + `AMT_CREDIT_SUM_LIMIT`
        - 87k cases where `AMT_CREDIT_SUM` != `AMT_CREDIT_SUM_DEBT` + `AMT_CREDIT_SUM_LIMIT`
        - 1310 cases where `AMT_CREDIT_SUM` = 0 but `AMT_CREDIT_SUM_DEBT` > 0
        - 63k have > `AMT_CREDIT_SUM` and either of the two missing

In [13]:
# Set AMT_CREDIT_SUM_LIMIT to 0 for closed credits where it is null
Bureau.loc[(Bureau['CREDIT_ACTIVE'] == 'Closed') & (Bureau['AMT_CREDIT_SUM_LIMIT'].isnull()), 'AMT_CREDIT_SUM_LIMIT'] = 0

In [14]:
# Set AMT_CREDIT_SUM_DEBT to 0 for closed credits where it is null
Bureau.loc[(Bureau['CREDIT_ACTIVE'] == 'Closed') & (Bureau['AMT_CREDIT_SUM_DEBT'].isnull()), 'AMT_CREDIT_SUM_DEBT'] = 0

In [ ]:
Bureau.loc[(Bureau['CREDIT_ACTIVE'] == 'Active') & 
           (Bureau['AMT_CREDIT_SUM_LIMIT'].isnull()) & 
           (Bureau['CREDIT_TYPE'] != 'Credit card'), 
           'AMT_CREDIT_SUM_LIMIT'] = 0

Calculate utilization rate for Credit cards to determine the average debit amount.

In [20]:
def get_utilization_rate(x, fulldf):
    y = fulldf.loc[x.index]
    y = y[y['CREDIT_ACTIVE'] == 'Active']
    return np.mean(np.where(
                            y['AMT_CREDIT_SUM'] > 0,
                            y['AMT_CREDIT_SUM_DEBT'] / y['AMT_CREDIT_SUM'],
                            np.nan
                        )
                    )

In [45]:
Bureau['CREDIT_ACTIVE'].value_counts()

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

# Bureau Balance

In [21]:
BureauBal.dtypes

SK_ID_BUREAU       int64
MONTHS_BALANCE     int64
STATUS            object
dtype: object

In [22]:
BureauBal = BureauBal.astype({'SK_ID_BUREAU': 'string'})

In [23]:
BureauBal.sort_values(by=['SK_ID_BUREAU', 'MONTHS_BALANCE'], inplace=True)

Impute data by going forward and backward.

In [24]:
BureauBal['STATUS_FILLED'] = BureauBal.groupby('SK_ID_BUREAU')['STATUS'].transform(
    lambda x: x.replace('X', np.nan).ffill(limit=3).bfill(limit=3)
)

C:\Users\domjo\AppData\Local\Temp\ipykernel_6352\2127701334.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.replace('X', np.nan).ffill(limit=3).bfill(limit=3)


In [25]:
BureauBal.groupby('SK_ID_BUREAU').last()#.reset_index()[['SK_ID_BUREAU', 'STATUS']].value_counts(dropna=False)

,MONTHS_BALANCE,STATUS,STATUS_FILLED
SK_ID_BUREAU,,,
5001709,0,C,C
5001710,0,C,C
5001711,0,X,0
5001712,0,C,C
5001713,0,X,None
...,...,...,...
6842884,0,C,C
6842885,0,5,5
6842886,0,C,C


In [42]:
BureauBal['STATUS_FILLED'].value_counts(dropna=False)

STATUS_FILLED
C      13735833
0       8705656
NaN     4488360
1        260330
5         67892
2         25590
3          9496
4          6768
Name: count, dtype: int64

In [71]:
BureauBal_DPD = BureauBal.groupby('SK_ID_BUREAU').agg(
    DURATION_BALANCE=('MONTHS_BALANCE', lambda x: x.max() - x.min()),
    BUREAUBAL_LAST_STATUS=('STATUS_FILLED', 'last'),
    BUREAUBAL_DAYS_WITH_NO_DPD=('STATUS_FILLED', lambda x: sum(x.isin(['0', 'C']))),
#    BUREAUBAL_DAYS_IN_DPD1=('STATUS_FILLED', lambda x: sum(x == '1')),
#    BUREAUBAL_DAYS_IN_DPD2=('STATUS_FILLED', lambda x: sum(x == '2')),
#    BUREAUBAL_DAYS_IN_DPD3=('STATUS_FILLED', lambda x: sum(x == '3')),
#    BUREAUBAL_DAYS_IN_DPD4=('STATUS_FILLED', lambda x: sum(x == '4')),
#    BUREAUBAL_DAYS_IN_DPD5=('STATUS_FILLED', lambda x: sum(x == '5')),
    BUREAUBAL_DAYS_IN_DPD_GE1 =('STATUS_FILLED', lambda x: sum(x.isin(['1', '2', '3', '4', '5'])))
).reset_index()

Days to DPD

In [27]:
BureauBal_DPDge1 = BureauBal[BureauBal['STATUS_FILLED'].isin(['1', '2', '3', '4', '5'])].groupby(['SK_ID_BUREAU']).first().reset_index()[['SK_ID_BUREAU','MONTHS_BALANCE']] .rename(columns={'MONTHS_BALANCE': 'MONTH_DPD_GE1'}) 

In [29]:
BureauBal_First_Snapshot= BureauBal.groupby(['SK_ID_BUREAU'])[['MONTHS_BALANCE']].first().reset_index()
BureauBal_DPDge1 = BureauBal_DPDge1.merge(BureauBal_First_Snapshot, 
                                  on=['SK_ID_BUREAU'], how='left')
BureauBal_DPDge1['DAYS_TO_DPD_GE1'] = BureauBal_DPDge1['MONTH_DPD_GE1'] - BureauBal_DPDge1['MONTHS_BALANCE']

Last 12 Months

In [35]:
BureauBal_DPD_1Y = BureauBal.groupby('SK_ID_BUREAU').tail(12)

In [38]:
BureauBal_DPD_1Y = BureauBal.groupby('SK_ID_BUREAU').tail(12).groupby('SK_ID_BUREAU').agg(
    BUREAUBAL_DAYS_WITH_NO_DPD_1Y=('STATUS_FILLED', lambda x: sum(x.isin(['0', 'C']))),
    BUREAUBAL_DAYS_IN_DPD1_1Y=('STATUS_FILLED', lambda x: sum(x == '1')),
    BUREAUBAL_DAYS_IN_DPD2_1Y=('STATUS_FILLED', lambda x: sum(x == '2')),
    BUREAUBAL_DAYS_IN_DPD3_1Y=('STATUS_FILLED', lambda x: sum(x == '3')),
    BUREAUBAL_DAYS_IN_DPD4_1Y=('STATUS_FILLED', lambda x: sum(x == '4')),
    BUREAUBAL_DAYS_IN_DPD5_1Y=('STATUS_FILLED', lambda x: sum(x == '5')),
    BUREAUBAL_DAYS_IN_DPD_GE1_1Y =('STATUS_FILLED', lambda x: sum(x.isin(['1', '2', '3', '4', '5'])))
).reset_index()

In [40]:
BureauBal_DPD = BureauBal_DPD.merge(
    BureauBal_DPDge1[['SK_ID_BUREAU', 'DAYS_TO_DPD_GE1']],
    how='left',
    on='SK_ID_BUREAU'
)

In [41]:
BureauBal_DPD = BureauBal_DPD.merge(
    BureauBal_DPD_1Y,
    how='left',
    on='SK_ID_BUREAU'
)

# Bureau

In [43]:
Bureau = Bureau.merge(
    BureauBal_DPD,
    how='left',
    on='SK_ID_BUREAU'
)

In [49]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [51]:
temp = Bureau[Bureau['ANNUITY_PERCENT'] > 2]
temp = temp.merge(base, on = 'SK_ID_CURR', how='left')

In [67]:
Bureau['UTILIZATION_RATIO'] = np.where(Bureau['AMT_CREDIT_SUM']>0, Bureau['AMT_CREDIT_SUM_DEBT'] / Bureau['AMT_CREDIT_SUM'], np.nan)
Bureau['CREDIT_SUM_OVERDUE_RATIO'] = np.where(Bureau['AMT_CREDIT_SUM']>0, Bureau['AMT_CREDIT_SUM_OVERDUE'] / Bureau['AMT_CREDIT_SUM'], np.nan)
Bureau['DURATION'] = Bureau['DAYS_CREDIT_ENDDATE'] - Bureau['DAYS_CREDIT'] 
Bureau['OVERDRAFT_FLAG'] = 1 * (Bureau['AMT_CREDIT_SUM_DEBT'] > Bureau['AMT_CREDIT_SUM'])
Bureau['ANNUITY_PERCENT'] = np.where(Bureau['AMT_CREDIT_SUM']>0, Bureau['AMT_ANNUITY'] / Bureau['AMT_CREDIT_SUM'], np.nan)

Average Bureau features

In [ ]:
# 1. Define your list of features
features_to_aggregate = [
    'UTILIZATION_RATIO', 
    'CREDIT_SUM_OVERDUE_RATIO',
    'AMT_CREDIT_MAX_OVERDUE', 
    'AMT_CREDIT_SUM',
    'AMT_CREDIT_SUM_DEBT',
    'ANNUITY_PERCENT',
    'DAYS_CREDIT_UPDATE'
    
]

# 2. Perform the aggregation
# This assumes 'df' is your dataframe and 'SK_ID_CURR' is the customer ID
Bureau_agg_df = Bureau.groupby('SK_ID_CURR')[features_to_aggregate].agg(['mean', 'max', 'min'])

In [70]:
# 3. Flatten the Multi-Index columns (Crucial for ML models)
# This changes columns from ('feat', 'mean') to 'feat_mean'
Bureau_agg_df.columns = ["BUREAU_"+'_'.join(col).strip().upper() for col in Bureau_agg_df.columns.values]
Bureau_agg_df = Bureau_agg_df.reset_index()

Other aggregations

In [ ]:
Bureau_agg_df_2 = Bureau.groupby('SK_ID_CURR').agg(

    BUREAU_TOTAL_CREDIT_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_CREDIT_TYPE_TOTAL_COUNT=('CREDIT_TYPE', 'nunique'),
    BUREAU_CONSUMERCREDIT_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Consumer credit')),
    BUREAU_CREDITCARD_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Credit card')),
    BUREAU_CARLOAN_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Car loan')),
    BUREAU_MORTGAGE_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Mortgages')),

    BUREAU_TOTAL_ACTIVE_CREDIT_COUNT=('SK_ID_BUREAU', lambda x: sum(Bureau.loc[x.index, 'CREDIT_ACTIVE'] == 'Active')),
    BUREAU_PAST_BAD_CREDIT_COUNT=('SK_ID_BUREAU', lambda x: sum(Bureau.loc[x.index, 'CREDIT_ACTIVE'] == 'Bad debt')),
    BUREAU_PAST_SOLD_CREDIT_COUNT=('SK_ID_BUREAU', lambda x: sum(Bureau.loc[x.index, 'CREDIT_ACTIVE'] == 'Sold')),

    BUREAU_DAYS_TO_LAST_CREDIT=('DAYS_CREDIT', 'max'),
    BUREAU_AVG_DAYS_TO_CREDIT=('DAYS_CREDIT', lambda x: x.sort_values(ascending=True).diff().mean() ), #"average time between credit applications
    BUREAU_AVG_DURATION=('DURATION', 'mean'),
    BUREAU_MAX_DAYS_CREDIT_END=('DAYS_CREDIT_ENDDATE', 'max'),
    BUREAU_OVERDRAFT_FLAG_COUNT = ('OVERDRAFT_FLAG', 'sum'),
    BUREAU_IS_PAST_OVERDUE_FLAG=('CREDIT_DAY_OVERDUE', lambda x: (x > 0).any() * 1),
    BUREAU_TOT_CREDIT_PROLONGATION_CNT = ('CNT_CREDIT_PROLONGATION', 'sum'),
    BUREAU_MAX_DAYS_CREDIT_UPDATE = ('DAYS_CREDIT_UPDATE', 'max') #when did last information about the Credit Bureau credit come?
)

In [29]:
Bureau_Silver.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\bureau_processed.parquet")